# PadChest preprocessing

PadChest has a richer taxonomy and serialized label formats. This notebook prints the observed vocabulary and matching examples before filtering. The exact target label must be verified in the downloaded release; no NIH/CheXpert label assumptions are made.

In [ ]:
from pathlib import Path
import json, sys, platform
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').is_dir() and Path('/content').exists(): PROJECT_ROOT = Path('/content') / 'Project Repo'
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from config import TARGET_DISEASE, RANDOM_SEED, MAX_POSITIVE_SAMPLES, MAX_NEGATIVE_SAMPLES
from dataset_utils import padchest_clean_binary_labels, padchest_find_label_vocabulary, standardize_padchest
from preprocessing_utils import (assert_no_leakage, assert_no_study_leakage, build_summary, normalize_label, parse_label_list, require_directory, require_file, sample_clean_cases, save_splits, save_summary, set_reproducibility, split_by_patient)
set_reproducibility(RANDOM_SEED)
print('Python:', platform.python_version(), 'pandas:', pd.__version__)
PADCHEST_ROOT = PROJECT_ROOT / 'data' / 'PadChest'
METADATA_CSV = PADCHEST_ROOT / 'PADCHEST_chest_x_ray_images_labels_160K_01.02.19.csv'
IMAGE_DIR = PADCHEST_ROOT / 'images'
OUTPUT_DIR = PROJECT_ROOT / 'processed' / 'client_03_PadChest'
print({'PADCHEST_ROOT': PADCHEST_ROOT, 'METADATA_CSV': METADATA_CSV, 'IMAGE_DIR': IMAGE_DIR, 'OUTPUT_DIR': OUTPUT_DIR})

In [ ]:
metadata_path = require_file(METADATA_CSV, 'PadChest metadata CSV')
image_dir = require_directory(IMAGE_DIR, 'PadChest image directory')
padchest = pd.read_csv(metadata_path)
print('images:', len(padchest))
print('unique patients:', padchest['PatientID'].nunique() if 'PatientID' in padchest else 'column unavailable')
print('unique studies:', padchest['StudyID'].nunique() if 'StudyID' in padchest else 'column unavailable')
print('columns:', padchest.columns.tolist())
display(padchest.isna().sum().sort_values(ascending=False).head(30).to_frame('missing'))
for column in ['ViewPosition_DICOM', 'Projection', 'PatientSex_DICOM', 'Manufacturer_DICOM']:
    if column in padchest: display(padchest[column].value_counts(dropna=False).head(20).to_frame(column))
if 'Labels' not in padchest.columns: raise KeyError('PadChest metadata must contain a Labels column')
vocabulary = padchest_find_label_vocabulary(padchest)
matches = [label for label in vocabulary if 'pleural' in normalize_label(label) or 'effusion' in normalize_label(label)]
print('Labels matching pleural/effusion:', matches)
if not matches: raise ValueError('No PadChest label matched Pleural Effusion. Inspect vocabulary and set the verified target label manually.')
TARGET_LABEL = next((label for label in matches if normalize_label(label) == normalize_label(TARGET_DISEASE)), matches[0])
print('Selected verified target label:', TARGET_LABEL)
display(padchest[padchest['Labels'].map(lambda value: any(normalize_label(x) == normalize_label(TARGET_LABEL) for x in parse_label_list(value)))][['ImageID','Labels']].head(10))

In [ ]:
clean, counts = padchest_clean_binary_labels(padchest, TARGET_LABEL)
print('Counts before filtering:', counts)
print('Clean negative policy: only an exact No Finding label is negative; all other non-target cases are excluded.')
standardized = standardize_padchest(clean, image_dir)
standardized = sample_clean_cases(standardized, MAX_POSITIVE_SAMPLES, MAX_NEGATIVE_SAMPLES, RANDOM_SEED)
display(standardized.head())
splits = split_by_patient(standardized, seed=RANDOM_SEED)
assert_no_leakage(splits)
assert_no_study_leakage(splits)
print('patient and study leakage checks: passed')
save_splits(splits, OUTPUT_DIR)
summary = build_summary(splits, 'PadChest', {'target_disease': TARGET_DISEASE, 'verified_target_label': TARGET_LABEL, 'random_seed': RANDOM_SEED, 'max_positive_samples': MAX_POSITIVE_SAMPLES, 'max_negative_samples': MAX_NEGATIVE_SAMPLES, 'filter_counts_before_filtering': counts})
save_summary(summary, OUTPUT_DIR)
print(json.dumps(summary, indent=2))
display(pd.concat(splits).groupby('target').size().rename('count').to_frame())

In [ ]:
# Final guard: a fuzzy vocabulary match must never become the experiment target silently.
if normalize_label(TARGET_LABEL) != normalize_label(TARGET_DISEASE):
    raise ValueError(
        f"PadChest has no exact label for {TARGET_DISEASE!r}. "
        f"Review the printed matches and set TARGET_LABEL explicitly after manual verification."
    )
